# Part 2. Structured Outputs
As we have seen in "Part 1: Prompt Engineering", typically LLMs output some sort of text, i.e. there is no structure in it.
In this section, we will explore different ways of and tools (**Pydantic** and **Instructor**) available for generating structured outputs in the form of **JSON** (JavaScript Object Notation) objects.
Don't worry if you don't know what a JSON object is - it will make sense once you've seen an example.

The main advantages of relying on structured outputs:
- generate (more) predictable responses;
- allow to build (more) robust AI systems;
- format the data so that it's ready for downstream tasks (e.g., passing an output resulting from one LLM call to an AI Agent).

In [1]:
# # Start our ollama server in the background to host our LLMs
from ollama_utils import start_ollama_server, stop_ollama_server

# start ollama server
start_ollama_server()

# model names
llama8b = "llama3.1:8b"
llama3b = "llama3.2:latest"

🚀 Starting Ollama server...
📄 Server logs: ollama_server.log
📍 API endpoint: http://localhost:11434
⏳ Waiting 5 seconds for server startup...
✅ Ollama server ready! PID: 2270628


In [2]:
import json
import textwrap
import instructor
from datetime import date
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Annotated, Literal, List

from IPython.display import JSON

In [3]:
# Create an OpenAI client
openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

#### Example 1: Structured outputs with information about scientists.

Imagine you would like an LLM to output information about a scientist in a specific JSON format:
```
{
    "name": "<scientist's name>",
    "occupation": "<scientist's occupation>",
    "date_of_birth": "<scientist's data of birth>",
    "facts": [
        <Fact 1>,
        <Fact 2>,
        ...
        <Fact n>
    ] 
}
```

In [4]:
# Let's define a data schema/model using Pydantic's BaseModel class.
# Inside our class we must provide field names and their types.

# Note: Pydantic has a Field object which allows us to provide more 
# configuration parameters such as additional information, metadata,
# and validation constraints.

class Scientist(BaseModel):
    name: str
    occupation: str
    date_of_birth: str
    facts: List[str] = Field(..., description="A list of facts about the scientist.")

In [5]:
# We can see the json schema of the Pydantic's model created above
# by calling the model_json_schema method.

code_generation_json_schema = json.dumps(
    Scientist.model_json_schema(),
    indent=2
)
print(code_generation_json_schema)

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "occupation": {
      "title": "Occupation",
      "type": "string"
    },
    "date_of_birth": {
      "title": "Date Of Birth",
      "type": "string"
    },
    "facts": {
      "description": "A list of facts about the scientist.",
      "items": {
        "type": "string"
      },
      "title": "Facts",
      "type": "array"
    }
  },
  "required": [
    "name",
    "occupation",
    "date_of_birth",
    "facts"
  ],
  "title": "Scientist",
  "type": "object"
}


In [6]:
# Now let's move to Instructor.
# Wrap the OpenaAI client into the Instructor client.

instructor_client = instructor.from_openai(
    openai_client,
    mode=instructor.Mode.JSON # When using with ollama, provide mode parameter 
)

In [7]:
# Important: The response will be of type Scientist! 

response = instructor_client.chat.completions.create(
    model=llama8b,
    messages=[{"role": "user", "content": "Tell me about Albert Einstein"}],
    response_model=Scientist, # provide Pydantic model here
    max_retries=3,
    temperature=0.7
)

print("Type: ", type(response))
print(response)

Type:  <class '__main__.Scientist'>
name='Albert Einstein' occupation='Theoretical Physicist and Mathematician' date_of_birth='March 14, 1879' facts=['Einstein developed the Theory of Relativity.', 'He is best known for his famous equation E=mc^2', 'He was awarded the Nobel Prize in Physics in 1921.']


In [8]:
# We can now access all the data generated by the LLM by providing an attribute name.

print("Name:", response.name)
print("Occupation", response.occupation)
print("Date of Birth:", response.date_of_birth)
print("*" * 100)

# Since facts are of type list, we can iterate over it
for i, f in enumerate(response.facts):
    print(f"Fact {i+1}:")
    print(f)

Name: Albert Einstein
Occupation Theoretical Physicist and Mathematician
Date of Birth: March 14, 1879
****************************************************************************************************
Fact 1:
Einstein developed the Theory of Relativity.
Fact 2:
He is best known for his famous equation E=mc^2
Fact 3:
He was awarded the Nobel Prize in Physics in 1921.


In [9]:
# We can also transform (serialize) the data from an instance of Scientist
# to JSON format.

response_json = response.model_dump_json()

print(response_json)
print(type(response_json))

{"name":"Albert Einstein","occupation":"Theoretical Physicist and Mathematician","date_of_birth":"March 14, 1879","facts":["Einstein developed the Theory of Relativity.","He is best known for his famous equation E=mc^2","He was awarded the Nobel Prize in Physics in 1921."]}
<class 'str'>


#### Example 2: Synthetic Data Generation
Let's imagine that we need to generate some records about users.

In [10]:
# Wrap the OpenaAI client into the Instructor client
instructor_client = instructor.from_openai(
    openai_client,
    mode=instructor.Mode.JSON # When using with ollama, provide mode parameter 
)

In [11]:
# Create a Pydantic model

class User(BaseModel):
    first_name: str = Field(..., description="user's first name")
    second_name: str = Field(..., description="user's second name")
    date_of_birth: date = Field(..., description="user's date of birth")
    address_number: str = Field(..., description="building number where user lives")
    street_name: str = Field(..., description="street where user lives")
    city_name: str = Field(..., description="city in which user lives")
    state: str = Field(..., description="state (within) the United States of America where user lives")
    zip_code: str = Field(..., description="zip code", min_length=5, max_length=5)

In [12]:
system_prompt = "You are a data generation system. Your task is to create content based on the specifications provided by the user."
user_prompt = "Create a random user's data matching the data schema provided."

In [14]:
generated_user_data = instructor_client.chat.completions.create(
    model=llama3b,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_model=User,
    max_retries=3,
    temperature=1.0
)

print("Type: ", type(generated_user_data))
print(generated_user_data)

Type:  <class '__main__.User'>
first_name='Ethan' second_name='Blackwood' date_of_birth=datetime.date(1992, 6, 1) address_number='1426' street_name='Oak Street' city_name='Atlanta' state='Georgia' zip_code='30324'


In [15]:
generated_user_data_json = generated_user_data.model_dump_json(indent=2)
print(generated_user_data_json)
print(f"Type: {type(generated_user_data_json)}")

{
  "first_name": "Ethan",
  "second_name": "Blackwood",
  "date_of_birth": "1992-06-01",
  "address_number": "1426",
  "street_name": "Oak Street",
  "city_name": "Atlanta",
  "state": "Georgia",
  "zip_code": "30324"
}
Type: <class 'str'>


In [16]:
# Let's generate more users
# Warning: There will be some cases when an LLM will fail to generate 
# a correct output.

users_list = []

for i in range(20):
    print(f"Iteration # {i+1}")
    
    try:
        generated_user = instructor_client.chat.completions.create(
            model=llama8b,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_model=User,
            max_retries=2,
            temperature=1.0
        )
        
        users_list.append(generated_user)
    
    except Exception as ex:
        print(f"Something went wrong at iteration # {i+1}")
        print(ex)

Iteration # 1
Iteration # 2
Iteration # 3
Iteration # 4
Iteration # 5
Iteration # 6
Something went wrong at iteration # 6
Failed to validate model User: 1 validation error for User
state
  Field required [type=missing, input_value={'address_number': '4323'...et_name': 'Bates Creek'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
Iteration # 7
Iteration # 8
Iteration # 9
Iteration # 10
Iteration # 11
Iteration # 12
Iteration # 13
Iteration # 14
Iteration # 15
Iteration # 16
Iteration # 17
Iteration # 18
Iteration # 19
Iteration # 20


In [17]:
users_list

[User(first_name='Olivia', second_name='Mcintosh', date_of_birth=datetime.date(1993, 4, 27), address_number='1450', street_name='Halsey Ave', city_name='Chicago', state='Illinois', zip_code='60008'),
 User(first_name='Jessica', second_name='Therese', date_of_birth=datetime.date(1994, 10, 20), address_number='23-B', street_name='Pine View Drive', city_name='Denver', state='Colorado', zip_code='80003'),
 User(first_name='Emily', second_name='Jane', date_of_birth=datetime.date(1995, 2, 12), address_number='1234', street_name='Main Street', city_name='San Francisco', state='California', zip_code='94111'),
 User(first_name='John', second_name='Alexander', date_of_birth=datetime.date(1985, 4, 12), address_number='1234', street_name='Elm street', city_name='New York', state='NY', zip_code='10001'),
 User(first_name='Cordelia', second_name="O'connor", date_of_birth=datetime.date(2002, 11, 12), address_number='KZS8 1K6Y', street_name='Nunley Crossing', city_name='Port Arthur', state='West Virgi

In [43]:
stop_ollama_server()

🛑 Ollama server stopped
